In [1]:
import json
from src.adapters import ProviderAdapter, AnthropicAdapter, OpenAIAdapter, DeepseekCoTAdapter
from dotenv import load_dotenv
import src.utils as utils
from src.models import ARCTaskOutput, ARCPair
from src.prompts.prompt_manager import convert_task_pairs_to_prompt
from typing import List, Any, Optional
import os
import argparse
import arckit

/Users/kaijie/workspace/arc/ARC-NLCoT/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# generation_model_name = "agentica-org/DeepScaleR-1.5B-Preview"
# generation_base_url = "http://100.95.78.36:8000/v1"
generation_model_name = "deepseek-reasoner"
generation_base_url = "https://api.deepseek.com"
extraction_model_name = "deepseek-reasoner"
extraction_base_url = "https://api.deepseek.com"
parse_cot = False

In [3]:
provider =DeepseekCoTAdapter(generation_model_name, generation_base_url, extraction_model_name, extraction_base_url, parse_cot)

In [10]:
task_id = "7c008303"
data_dir = "data/arc-agi/data/training"

In [11]:
train_pairs = utils.get_train_pairs_from_task(data_dir, task_id)
test_input = utils.get_test_input_from_task(data_dir, task_id)
test_input_pair = test_input[0]

In [14]:
print(test_input)

[ARCPair(input=[[0, 0, 0, 3, 0, 0, 8, 0, 0], [3, 3, 0, 3, 0, 3, 8, 0, 0], [0, 3, 0, 3, 0, 3, 8, 0, 0], [0, 3, 3, 3, 0, 0, 8, 0, 0], [0, 3, 0, 0, 0, 3, 8, 0, 0], [0, 0, 3, 0, 0, 0, 8, 0, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8], [0, 0, 0, 0, 0, 0, 8, 2, 1], [0, 0, 0, 0, 0, 0, 8, 4, 7]], output=None)]


## Generate Prompts

In [13]:
prompt = convert_task_pairs_to_prompt(train_pairs, test_input_pair)
print(prompt)

You are participating in a puzzle solving competition. You are an expert at solving puzzles.

Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation in the training examples that maps the input to the output, then apply that pattern to the test input to give a final output.

Respond in the format of the training output examples

--Training Examples--
--Example 0-- 

 INPUT: 

[2, 4, 8, 0, 0, 0, 0, 0, 0]
[1, 6, 8, 0, 0, 0, 0, 0, 0]
[8, 8, 8, 8, 8, 8, 8, 8, 8]
[0, 0, 8, 0, 3, 0, 0, 3, 0]
[0, 0, 8, 3, 3, 3, 3, 3, 3]
[0, 0, 8, 0, 3, 0, 0, 3, 0]
[0, 0, 8, 0, 3, 0, 0, 3, 0]
[0, 0, 8, 3, 3, 3, 3, 3, 3]
[0, 0, 8, 0, 3, 0, 0, 3, 0]


OUTPUT: 

[0, 2, 0, 0, 4, 0]
[2, 2, 2, 4, 4, 4]
[0, 2, 0, 0, 4, 0]
[0, 1, 0, 0, 6, 0]
[1, 1, 1, 6, 6, 6]
[0, 1, 0, 0, 6, 0]


--Example 1-- 

 INPUT: 

[0, 0, 0, 0, 0, 0, 8, 1, 2]
[0, 0, 0, 0, 0, 0, 8, 4, 1]
[8, 8, 8, 8, 8, 8, 8, 8, 8]
[0, 0, 3, 3, 0, 3, 8, 0, 0]
[3, 3, 0, 0, 0, 0, 8, 0, 0]
[3, 3, 0, 3, 0, 3

In [10]:
print(test_input)

[ARCPair(input=[[6, 3, 5], [6, 8, 0], [4, 0, 0]], output=None)]


## Visualize task

In [6]:
task = arckit.load_single(task_id)